# OpenMontage Stage 1: Prepare Colab Runtime

Install minimal system and Python dependencies required to run the *real* Stage 1 pipeline on a Colab GPU. Run each cell in order.
Use a GPU runtime (Runtime -> Change runtime type -> GPU).

In [ ]:
# 1) Install system packages (ffmpeg)
import subprocess
print('Installing apt packages: ffmpeg')
subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','ffmpeg','libsndfile1'], check=True)
print('ffmpeg installed')


In [ ]:
# 2) Pin setuptools to avoid PyTorch compatibility issue, install jedi for ipython, and upgrade pip
import subprocess, sys
print('Pinning setuptools to 81.8.0 and installing jedi')
subprocess.run([sys.executable,'-m','pip','install','-U','pip'], check=True)
subprocess.run([sys.executable,'-m','pip','install','setuptools==81.8.0','jedi'], check=True)
print('pip/setuptools/jedi done')


In [ ]:
# 3) Install Python packages needed by the minimal real pipeline
# faster-whisper (transcription), transformers (music), diffusers (image fallback), moviepy (montage fallback), scipy
import subprocess, sys
pkgs = [
    'faster-whisper',
    'transformers',
    'diffusers',
    'accelerate',
    'safetensors',
    'moviepy',
    'scipy',
    'torch',
    'torchaudio'
]
print('Installing packages (this may take several minutes):', pkgs)
subprocess.run([sys.executable,'-m','pip','install','--upgrade'] + pkgs, check=False)
print('Package install attempted; inspect output for failures')


Note: Installing torch via plain pip will select the CPU wheel by default. If the earlier GPU check showed a GPU but torch.cuda.is_available() is False, prefer installing a CUDA wheel manually (see the check notebook). If pip installed a CPU-only torch, restart the runtime and install the CUDA wheel described in the check notebook.

In [ ]:
# 4) Clone repository and set PYTHONPATH for this notebook session
import os, subprocess, sys
repo_dir = '/content/openmontage-colab'
if not os.path.exists(repo_dir):
    print('Cloning repository into', repo_dir)
    subprocess.run(['git','clone','--depth','1','https://github.com/z3685507-tech/openmontage-colab.git',repo_dir], check=True)
else:
    print('Repository already exists at', repo_dir)
os.environ['PYTHONPATH'] = repo_dir + ':' + os.environ.get('PYTHONPATH','')
print('PYTHONPATH set to', os.environ['PYTHONPATH'])
os.chdir(repo_dir)
print('CWD now', os.getcwd())


5) After running the above cells:
- Restart the runtime if you installed or changed torch wheels.
- Then run the `docs/colab_stage1_check.ipynb` notebook's final cell to run the strict smoke test.

If you prefer, run the smoke runner directly from this notebook with the next cell.

In [ ]:
# 6) (Optional) Run strict smoke runner from here
import os, subprocess, sys
confirm = 'yes'
if confirm.lower() == 'yes':
    os.environ['OM_LOAD_ADAPTERS'] = '1'
    os.environ['OM_REAL_STRICT'] = '1'
    print('Running strict smoke runner...')
    r = subprocess.run([sys.executable,'tools/run_stage1_smoke.py'], capture_output=True, text=True)
    print('Return code:', r.returncode)
    print(r.stdout)
    print(r.stderr)
else:
    print('Set confirm="yes" to run the smoke runner')
